In [1]:
from qwen3_moe_fork.modeling_qwen3_moe import Qwen3MoeConfig, Qwen3MoeForCausalLM

W0930 23:44:44.526000 6930 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0930 23:44:44.544000 6930 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0930 23:44:44.563000 6930 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [12]:
QWEN3_MOE_BASELINE_CONFIG = dict(
    vocab_size=50254,
    hidden_size=512,
    intermediate_size=6144,
    num_hidden_layers=12,
    num_attention_heads=8,
    num_key_value_heads=1,
    hidden_act="silu",
    max_position_embeddings=2048,
    initializer_range=0.02,
    rms_norm_eps=1e-6,
    use_cache=True,
    tie_word_embeddings=False,
    rope_parameters=None,
    attention_bias=False,
    use_sliding_window=False,
    sliding_window=4096,
    attention_dropout=0.0,
    decoder_sparse_step=1,
    moe_intermediate_size=352,
    num_experts_per_tok=3,
    num_experts=48,
    norm_topk_prob=False,
    output_router_logits=True,
    router_aux_loss_coef=0.001,
    mlp_only_layers=None,
    pad_token_id=1,
    bos_token_id=0,
    eos_token_id=0,
)

In [14]:
config = Qwen3MoeConfig(**QWEN3_MOE_BASELINE_CONFIG)

In [15]:
model = Qwen3MoeForCausalLM(config)

In [16]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

total_params = count_params(model)
embed_params = count_params(model.model.embed_tokens)
lm_head_params = count_params(model.lm_head)

print(f"Total params: {total_params/1e6:.1f}M")
print(f"Embedding table: {embed_params/1e6:.1f}M")
print(f"LM head: {lm_head_params/1e6:.1f}M")

Total params: 370.3M
Embedding table: 25.7M
LM head: 25.7M


In [17]:
attn_total = 0
router_total = 0
expert_total = 0

for layer in model.model.layers:
    attn_total += count_params(layer.self_attn)
    router_total += count_params(layer.mlp.gate)
    expert_total += count_params(layer.mlp.experts)

top_k = config.num_experts_per_tok
num_experts = config.num_experts
expert_active = expert_total * (top_k / num_experts)

active_total = embed_params + lm_head_params + attn_total + router_total + expert_active

print(f"Attention (all layers): {attn_total/1e6:.1f}M")
print(f"Router weights (all layers): {router_total/1e6:.1f}M")
print(f"Experts, ALL stored: {expert_total/1e6:.1f}M")
print(f"Experts, ACTIVE ({top_k}/{num_experts}): {expert_active/1e6:.1f}M")
print(f"\nEstimated ACTIVE params per token: {active_total/1e6:.1f}M")

Attention (all layers): 7.1M
Router weights (all layers): 0.3M
Experts, ALL stored: 311.4M
Experts, ACTIVE (3/48): 19.5M

Estimated ACTIVE params per token: 78.3M


In [18]:
import torch

batch_size, seq_len = 2, 16
input_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))

model.eval()
with torch.no_grad():
    out = model(input_ids=input_ids, labels=input_ids)

print("logits shape:", out.logits.shape, "-> expected:", (batch_size, seq_len, config.vocab_size))
print("loss (CE + aux):", out.loss.item())
print("aux loss alone:", out.aux_loss.item())

logits shape: torch.Size([2, 16, 50254]) -> expected: (2, 16, 50254)
loss (CE + aux): 10.88250732421875
aux loss alone: 3.1041319370269775


In [19]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
print("vocab size:", tokenizer.vocab_size)
print("pad_token_id:", tokenizer.pad_token_id)
print("bos_token_id:", tokenizer.bos_token_id)
print("eos_token_id:", tokenizer.eos_token_id)

vocab size: 50254
pad_token_id: 1
bos_token_id: 0
eos_token_id: 0


In [20]:
print(config.pad_token_id, config.bos_token_id, config.eos_token_id)

1 0 0


In [21]:
total_params = count_params(model)
embed_params = count_params(model.model.embed_tokens)
lm_head_params = count_params(model.lm_head)
print(f"Total: {total_params/1e6:.1f}M  |  Embed: {embed_params/1e6:.1f}M  |  LM head: {lm_head_params/1e6:.1f}M")

Total: 370.3M  |  Embed: 25.7M  |  LM head: 25.7M


In [1]:
!pwd && ls

/content
sample_data
